# QuantumTrain 工具書 01：從量子機率到可訓練權重

閱讀與動手並重；預設 smoke。每節提供目標、理由、shape、操作、判讀與練習。

## 0. 閱讀地圖與操作模式

**本節目標**：知道閱讀顺序與執行預算。

**為什麼需要這一步**：先固定環境與模式，才能分辨程式檢查和科學實驗。

**輸入／輸出與 shape**：設定字典 → MODE 與結果目錄；此節尚不訓練模型。

**關鍵原理與程式定位**：流程：資料 → 量子機率＋bit 編碼 → Mapping → MLP 權重 → logits → loss。前置知識：Python 函式、張量、基本反向傳播；量子基礎會在第 4 節示範。

**操作步驟**：從專案執行 uv run jupyter lab，選目前 .venv 的 Python kernel；依序執行。第一次保持 smoke。重跑前用 Restart Kernel and Run All。

**預期觀察**：印出 smoke、cpu、float64。完整第一輪時間將由第 7 節的小型實測估計。

**結果判讀與常見錯誤**：只重新執行中段可能沿用舊模型或舊設定；已保存輸出不代表本次執行。

**理解問題**：為什麼預設不啟動 full？

<details><summary>提示與解答</summary>

全新讀者應先確認環境與梯度，再投入正式運算；smoke 結果不能作為結論。

</details>


In [ ]:
import os, sys, json, time
from pathlib import Path
import torch, pennylane as qml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from torch.utils.data import DataLoader
from qt_eval.config import load_config, validate_config
from qt_eval.data import load_digits_splits
from qt_eval.models import make_model, ClassicalMLP, functional_mlp
from qt_eval.quantum import QuantumTrain, make_qnode, basis_codes, probability_features
from qt_eval.diagnostics import check_training_step
from qt_eval.experiment import run_experiment, evaluate
from qt_eval.reporting import summarize_runs, write_report
from qt_eval.artifacts import SOURCE_COMMIT

# 教學預設 smoke；只有準備好正式預算後才手動改成 full。
MODE = os.environ.get('QT_EVAL_MODE', 'smoke')
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '請由專案根目錄或 notebooks 目錄啟動'
OUTPUT = Path(os.environ.get('QT_EVAL_OUTPUT', str(ROOT/'results'/'notebooks')))
CONFIG = load_config(ROOT/'configs'/('smoke.json' if MODE=='smoke' else 'depth_study.json'))
assert CONFIG['mode'] == MODE
torch.set_num_threads(CONFIG['torch_threads'])
torch.manual_seed(CONFIG['seed'])
print('MODE:', MODE, '| device: cpu | dtype: float64')
if MODE == 'smoke':
    print('工程檢查：小資料、短訓練，不可用來判斷模型研究表現。')

## 1. 環境、來源與改動

**本節目標**：辨認方法來源和這個作品自己的工作。

**為什麼需要這一步**：框架版本與改動會影響再現性；來源標記不能只寫「用了量子」。

**輸入／輸出與 shape**：套件與文件 → 版本表及原始碼參照。

**關鍵原理與程式定位**：docs/provenance.md 記錄論文、固定 commit 與獨立適配；目前是 digits/MLP，不是原論文數值重現。

**操作步驟**：檢查版本及 Python 路徑，打開來源說明。

**預期觀察**：Python 位於本專案 .venv，PennyLane 與 Torch 可匯入。

**結果判讀與常見錯誤**：安裝在一個環境卻用另一個 kernel，會出現找不到 qt_eval；先核對 sys.executable。

**理解問題**：改用 digits 是否仍能稱為 CIFAR-10 結果重現？

<details><summary>提示與解答</summary>

不能；保留架構思想但改變任務，應稱適配實驗。

</details>


In [ ]:
print(sys.executable)
print({'torch':torch.__version__, 'pennylane':qml.__version__, 'source_commit':SOURCE_COMMIT})
display(Markdown((ROOT/'docs/provenance.md').read_text(encoding='utf-8')))

## 2. 資料、圖像與切分

**本節目標**：看懂原始圖像如何變成模型輸入。

**為什麼需要這一步**：資料洩漏會使比較失去可信度，必須先於訓練檢查。

**輸入／輸出與 shape**：8×8 圖 → (N,64) float64；標籤 (N,) long。

**關鍵原理與程式定位**：data.py 固定除以 16，分層 60/20/20；split seed 固定 2026。

**操作步驟**：印出各 split 數量，確認交集為空，再看第一張圖。

**預期觀察**：smoke 每 split 40 筆，圖像像素在 [0,1]。

**結果判讀與常見錯誤**：不能先看 test 再選模型。實驗 seed 改初始化，split seed 改資料分割，兩者作用不同。

**理解問題**：為什麼除以 16 不需要 fit？

<details><summary>提示與解答</summary>

16 是資料已知的像素上限，不是從完整資料估計出的統計量。

</details>


In [ ]:
splits=load_digits_splits(CONFIG['split_seed'],smoke=MODE=='smoke')
print({k:len(splits[k]) for k in ('train','validation','test')})
index_sets=[set(splits['indices'][k]) for k in ('train','validation','test')]
assert all(not index_sets[i]&index_sets[j] for i in range(3) for j in range(i))
x,y=splits['train'].tensors
plt.imshow(x[0].reshape(8,8),cmap='gray',vmin=0,vmax=1)
plt.title(f'Digits train example; label={int(y[0])}')
plt.axis('off'); plt.show()
print(x.shape,y.shape,x.dtype,y.dtype)

## 3. 傳統 MLP 與 610 個參數

**本節目標**：手算參數量並確認兩種 forward 等價。

**為什麼需要這一步**：QT 只改權重來源；下游 MLP 的結構必須相同，才能有意義地比較。

**輸入／輸出與 shape**：(B,64) → (B,8) → (B,10)；B 是 batch size。

**關鍵原理與程式定位**：models.py：64×8+8+8×10+10=610；CrossEntropyLoss 接 logits，不要先加 softmax。

**操作步驟**：建立 baseline，flatten 權重，再對照 functional_mlp 的 logits。

**預期觀察**：參數數量 610，兩個輸出的最大差為零或數值誤差內。

**結果判讀與常見錯誤**：重新建立 nn.Parameter 或 .data.copy_ 容易切開生成器的計算圖，切片與 reshape 才保留連結。

**理解問題**：ReLU 應放在哪裡？

<details><summary>提示與解答</summary>

第一個線性層後；最後層維持 logits。兩個比較組要一致。

</details>


In [ ]:
baseline=ClassicalMLP().double()
flat=torch.cat([p.flatten() for p in baseline.parameters()])
print('Target parameters:',flat.numel())
print('Maximum difference:',float((baseline(x[:4])-functional_mlp(x[:4],flat)).abs().max().detach()))
print('Logits shape:',baseline(x[:4]).shape)

## 4. 小電路與量測機率

**本節目標**：理解 H、RY/RZ、CZ 與量測的關係。

**為什麼需要這一步**：10 qubits 太大，不適合第一次看電路；先用 2 qubits 觀察 4 個機率。

**輸入／輸出與 shape**：theta (L,2) → probabilities (4,)；L 是旋轉/CZ 重複層數。

**關鍵原理與程式定位**：quantum.py 的 make_qnode：H 產生疊加，RY 改變振幅，RZ/CZ 改變相位；末端純相位操作不改計算基底機率。

**操作步驟**：畫一層電路，比較追加 RZ/CZ 層前後機率。

**預期觀察**：機率和約 1，追加末端相位層前後一致；正式比較用 1/3/5，末層為 RY。

**結果判讀與常見錯誤**：相位可能影響之後的旋轉；不能推論 RZ 在所有位置都沒用。這裡的層數不是硬體編譯深度。

**理解問題**：若末端 RZ 的梯度是零，能直接說程式壞了嗎？

<details><summary>提示與解答</summary>

不能。此量測對該參數不敏感；這與 grad=None 的中斷不同。

</details>


In [ ]:
small=make_qnode(2)
theta=torch.tensor([[0.3,-0.7]],dtype=torch.float64,requires_grad=True)
print(qml.draw(small)(theta))
fig,ax=qml.draw_mpl(small)(theta); plt.show()
with_phase=torch.tensor([[0.3,-0.7],[0.9,0.4]],dtype=torch.float64)
print('Before:',small(theta).detach().numpy())
print('After: ',small(with_phase).detach().numpy())
torch.testing.assert_close(small(theta),small(with_phase))

## 5. 從機率到 MLP 權重

**本節目標**：沿著 shape 追蹤生成器。

**為什麼需要這一步**：量子位元不是一個對一個儲存 CNN/MLP 權重；這是用機率特徵生成權重的架構。

**輸入／輸出與 shape**：(L,10) → (1024,) → (610,1)；bit 編碼 (610,10)；合併 (610,11) → (610,1) → (610,)。

**關鍵原理與程式定位**：2^9<610≤2^10，因此需 10 qubits。f(p)=(0.8 tanh(0.1×512×p²))^0.3 再中心化。p 已是機率；平方是來源的额外轉換。

**操作步驟**：逐步印出 shape 與第 0 個權重。mapping 全為 Linear，16,761 個參數。

**預期觀察**：生成的第 0 個數值就是 fc1.weight[0,0]；其值保有 grad_fn。

**結果判讀與常見錯誤**：前 610 個機率未重新正規化，只作特徵。多層 affine 可合併為 affine；生成器比 610 大，本例沒有壓縮優勢。

**理解問題**：可訓練參數與生成的權重，為什麼數量不同？

<details><summary>提示與解答</summary>

前者是 optimizer 更新的來源；後者由來源計算得到，是 target forward 使用的張量。

</details>


In [ ]:
qt=QuantumTrain(CONFIG['q_depth'],CONFIG['q_delta'])
p=qt.qnode(qt.q_params)
f=probability_features(p[:610],10)
combined=torch.cat([qt.codes,f[:,None]],dim=1)
generated=qt.mapping(combined).flatten()
print({'probabilities':tuple(p.shape),'features':tuple(f.shape),'combined':tuple(combined.shape),'weights':tuple(generated.shape)})
print('First bit code / raw p / feature:',qt.codes[0],p[0].detach(),f[0].detach())
print('fc1.weight[0,0]:',generated[0].detach(),'grad_fn:',generated.grad_fn)
print('Mapping parameters:',sum(p.numel() for p in qt.mapping.parameters()))

## 6. 梯度連接與 detach 反例

**本節目標**：能分辨量子參數、梯度與生成權重。

**為什麼需要這一步**：loss 下降可能只因 mapping 在更新，必須觀察量子分支。

**輸入／輸出與 shape**：loss scalar → logits → generated weights → mapping → probabilities → q_params (L,10)。

**關鍵原理與程式定位**：diagnostics.py 在副本跑單步，保留正式模型與 RNG；detach 反例故意切斷 qnode 輸出。

**操作步驟**：比較正常和 detached 副本的診斷；不要對正式 qt 設 detach_quantum=True。

**預期觀察**：正常 quantum/mapping 均有梯度與 delta；反例 quantum.has_grad=False，mapping 仍有梯度。

**結果判讀與常見錯誤**：requires_grad=True 不是計算圖連接證明；有非零梯度也不是量子優勢。有限差分測試見 tests/test_core.py。

**理解問題**：把 detached tensor 的 requires_grad 設為 True 能接回去嗎？

<details><summary>提示與解答</summary>

不能；它只是新的 leaf，並沒有到原量子參數的鏈式法則。

</details>


In [ ]:
import copy
healthy=check_training_step(qt,x[:20],y[:20])
broken=copy.deepcopy(qt)
broken.detach_quantum=True
disconnected=check_training_step(broken,x[:20],y[:20])
print('Healthy:',healthy)
print('Detached:',disconnected)
assert healthy['status']=='passed' and disconnected['status']=='failed'

## 7. 最小訓練與成本預估

**本節目標**：親自看到參數更新與梯度大小。

**為什麼需要這一步**：在完整訓練前以一小步驗證回饋，也取得大致計時。

**輸入／輸出與 shape**：(最多32,64) batch → scalar loss；step 前後 theta 相同 shape。

**關鍵原理與程式定位**：Adam 使用 loss.backward() 的梯度更新；此小例子是正式模型的副本。

**操作步驟**：印出 loss、quantum grad norm、最大參數改變與時間。

**預期觀察**：有效參數會改變，估算只含 train step，不含 validation 與 I/O。

**結果判讀與常見錯誤**：第一次執行有 warm-up；單 batch 估計非硬體 benchmark，更非準確總時間。

**理解問題**：怎麼估算 smoke 的 train 時間？

<details><summary>提示與解答</summary>

步數×每步耗時只是粗估；還需模型初始化、驗證、測試與寫檔時間。

</details>


In [ ]:
demo=copy.deepcopy(qt)
optimizer=torch.optim.Adam(demo.parameters(),lr=CONFIG['learning_rate'])
before=demo.q_params.detach().clone()
start=time.perf_counter()
loss=torch.nn.functional.cross_entropy(demo(x[:32]),y[:32])
loss.backward()
gradient_norm=float(demo.q_params.grad.norm())
optimizer.step()
step_seconds=time.perf_counter()-start
print({'loss':float(loss.detach()),'quantum_grad_norm':gradient_norm,'max_delta':float((demo.q_params.detach()-before).abs().max()),'step_seconds':step_seconds})
steps=((len(x)+CONFIG['batch_size']-1)//CONFIG['batch_size'])*CONFIG['epochs']
print('Approximate train-only seconds:',steps*step_seconds,'(batch size differs; rough estimate)')

## 8. 訓練、驗證與測試

**本節目標**：執行完整且無 test 選模的流程。

**為什麼需要這一步**：单步成功不能代替完整訓練；驗證曲線能發現泛化與最佳 epoch 差異。

**輸入／輸出與 shape**：CONFIG → unique run 目錄；metrics 每 epoch 一列。

**關鍵原理與程式定位**：experiment.fit 只接 train/validation，最低 validation loss 保存 best.pt；run_experiment 載入後才評估 test。

**操作步驟**：執行當前設定，查看原始紀錄並畫 loss。

**預期觀察**：run status completed，兩條曲線有 epoch 軸；smoke 不要求 loss 單調下降或高 accuracy。

**結果判讀與常見錯誤**：訓練 loss 下滑且 validation 上升可能過擬合；小 smoke 資料不能判定方法有效。

**理解問題**：為什麼不用 test accuracy 挑 best epoch？

<details><summary>提示與解答</summary>

挑選過程會利用 test 資訊，最後 accuracy 不再是獨立評估。

</details>


In [ ]:
run=run_experiment(CONFIG,OUTPUT/'workbook')
history=pd.read_csv(run/'metrics.csv')
display(history)
ax=history.plot(x='epoch',y=['train_loss','val_loss'],marker='o')
ax.set_ylabel('Mean cross entropy'); plt.show()
print('Caption: each point is a sample-weighted epoch loss; validation does not update parameters.')
result=json.loads((run/'result.json').read_text())
print(run,result)

## 9. 保存與重載

**本節目標**：找到 artifact 並重建相同預測。

**為什麼需要這一步**：報告數字需要能回到具體模型和設定。

**輸入／輸出與 shape**：config.json＋best.pt → 同一組 test logits 與 metrics。

**關鍵原理與程式定位**：checkpoint 保存 state_dict；權重不是整個 pickled model，讀取自有產物使用 weights_only=True。

**操作步驟**：重新建立模型、載入 best.pt，重新計算固定 test 結果核對。此處是再現確認，不能據此改參數。

**預期觀察**：重載後 test loss/accuracy 與當次 result 一致。

**結果判讀與常見錯誤**：best.pt 不含 optimizer/RNG 狀態，不能稱為精確中斷續訓。

**理解問題**：只保存一個 accuracy 數字足夠嗎？

<details><summary>提示與解答</summary>

不足；還需要 split、設定、來源版本、checkpoint 與紀錄才能追溯。

</details>


In [ ]:
print([p.name for p in run.iterdir()])
loaded=make_model(CONFIG)
checkpoint=torch.load(run/'best.pt',weights_only=True)
loaded.load_state_dict(checkpoint['model_state'])
repeated=evaluate(loaded,DataLoader(splits['test'],batch_size=CONFIG['batch_size']))
assert repeated==result['test']
print('Reload verified:',repeated,'best epoch:',checkpoint['epoch'])

## 10. 小結與下一個實驗

**本節目標**：把執行成功轉為可驗證的敘述。

**為什麼需要這一步**：要能說明證據的範圍，而不是把量子名詞當成結論。

**輸入／輸出與 shape**：診斷＋曲線＋版本 → 一份可追溯的學習摘要。

**關鍵原理與程式定位**：研究延伸在 notebook 02；學習變因固定後才比較深度。

**操作步驟**：閱讀方法限制，試著用自己的話描述三種參數，再進入第二份教材。

**預期觀察**：能指出量子梯度健康檢查、目標參數 610、mapping 16,761，以及 smoke 的限制。

**結果判讀與常見錯誤**：本輪不是壓縮或 QPU 加速實驗。不要因成功跑通就宣稱量子優勢。

**理解問題**：最直接的下一個消融是什麼？

<details><summary>提示與解答</summary>

固定量子參數、只訓練 mapping；比較時維持相同初始化、資料與訓練預算。

</details>


In [ ]:
display(Markdown((ROOT/'docs/methodology.md').read_text(encoding='utf-8')))
print('Next: notebooks/02_depth_evaluation.ipynb')